# Crowd Predictor Framework - Model Training
Run this notebook to orchestrate training for both the LSTM neural network and Prophet forecasting models based on the raw telemetry dataset.

In [2]:
import pandas as pd
import os
import sys
import torch
import pickle
from prophet.serialize import model_to_json

# Change working directory to project root if running inside notebooks/
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from backend.app.models.lstm_model import CrowdLSTMModel
from backend.app.models.prophet_model import ForecastModel

# Ensure directories
os.makedirs('models', exist_ok=True)

print('Loading dataset...')
df = pd.read_csv('data/processed/processed_crowd_data.csv')
df['ds'] = pd.to_datetime(df['timestamp'])
df['y'] = df['count'].rolling(window=3, min_periods=1).mean()
df['hour'] = df['ds'].dt.hour
df['day'] = df['ds'].dt.dayofweek

# Use more data if possible
df_train = df.tail(15000).copy()
print(f'Training on {len(df_train)} datapoints.')


Loading dataset...
Training on 8755 datapoints.


## 1. Train & Export Prophet Model
Prophet utilizes robust statistical mechanics to establish seasonal trajectories based on the time series.

In [3]:
prophet = ForecastModel()
print('Training Prophet...')
prophet.train(df_train[['ds', 'y', 'hour', 'day']])

# Save Prophet Model
with open('models/prophet_model.json', 'w') as f:
    f.write(model_to_json(prophet.model))
print('Prophet model saved successfully to models/prophet_model.json')


Training Prophet...
Prophet model saved successfully to models/prophet_model.json


## 2. Train & Export LSTM Model
The LSTM module catches high-frequency fluctuations utilizing a neural gradient network architecture.

In [4]:
lstm = CrowdLSTMModel(sequence_length=60, epochs=300, lr=0.0005)
print('Training LSTM...')
lstm.train(df_train[['ds', 'y', 'hour', 'day']])

# Save LSTM Network Weights
torch.save(lstm.model.state_dict(), 'models/lstm_weights.pth')

# Save Scaler
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(lstm.scaler, f)
    
print('LSTM weights and scaling architecture saved successfully to models/')


Training LSTM...


KeyboardInterrupt: 

## 3. Model Evaluation
Perform a simple sanity check evaluation to compute Mean Absolute Error (MAE) and accuracy on the trailing data.

In [5]:
import numpy as np

print('--- Model Evaluation (Training Set Sanity Check) ---')
try:
    # Prophet evaluation (In-sample comparison)
    p_eval_df = df_train[['ds', 'hour', 'day']].copy()
    p_forecast_full = prophet.model.predict(p_eval_df)
    p_mae = np.abs(p_forecast_full['yhat'].values - df_train['y'].values).mean()
    p_acc = 100 * (1 - p_mae / df_train['y'].mean())
    print(f'Prophet - MAE: {p_mae:.2f}, Approximate Accuracy: {p_acc:.2f}%')
    
    # LSTM evaluation (1-step ahead on last 160 points)
    lstm.model.eval()
    features = ['y', 'hour', 'day']
    test_data = df_train[features].tail(160).values
    test_data_norm = lstm.scaler.transform(test_data)
    
    preds = []
    with torch.no_grad():
        for i in range(100):
            seq = torch.FloatTensor(test_data_norm[i:i+60]).unsqueeze(0)
            pred = lstm.model(seq).item()
            preds.append(pred)
            
    preds_padded = np.zeros((len(preds), 3))
    preds_padded[:, 0] = preds
    preds_orig = lstm.scaler.inverse_transform(preds_padded)[:, 0]
    
    actuals = test_data[60:, 0].flatten()
    l_mae = np.abs(preds_orig - actuals).mean()
    l_acc = 100 * (1 - l_mae / actuals.mean())
    print(f'LSTM    - MAE: {l_mae:.2f}, Approximate Accuracy: {l_acc:.2f}%')
    
except Exception as e:
    print(f'Could not complete evaluation: {e}')


--- Model Evaluation (Training Set Sanity Check) ---
Prophet - MAE: 12.13, Approximate Accuracy: 75.50%
LSTM    - MAE: 2.15, Approximate Accuracy: 93.19%
